# B2-023-generative-models-diffusion — Practice p11 — Solution

**Type:** constrained-coding · **Difficulty:** core · **Concepts:** denoising-diffusion-probabilistic-models

*50 minutes.*  
**Set:** B  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261015`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C11-neural-training`, `B2-019-attention-transformers`, `B2-020-language-transformers`, `B2-021-cross-modal-transformers-vision`, `B2-022-probabilistic-latent-models`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb), [B2-020-language-transformers](../../B2-020-language-transformers/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb), [B2-022-probabilistic-latent-models](../../B2-022-probabilistic-latent-models/lesson.ipynb).

**Data boundary:** CPU only; use literal unit data, the seeded unit generator, and standard library/NumPy/Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints, or student data.

## Supplied cell (copied verbatim from the statement)

In [ ]:
import math
import torch
from torch import nn

XT2 = torch.tensor([[0.4, -0.8], [1.2, 0.0]])
Z2 = torch.tensor([[1.0, -1.0], [0.5, 2.0]])
X_T = torch.tensor([[1.0, -1.0], [0.0, 2.0]])
NOISES = torch.tensor([[[0.1, 0.2], [0.3, -0.1]],
                       [[-0.5, 0.4], [0.2, 0.2]],
                       [[1.0, -1.0], [0.0, 0.5]],
                       [[0.3, 0.3], [-0.3, 0.1]],
                       [[-0.2, 0.6], [0.4, -0.4]]])


class ToyEps(nn.Module):
    def forward(self, x_t, t_scaled):
        return 0.5 * x_t + t_scaled


ATOL = 1e-5
RTOL = 1e-5

## Solution

`p_sample` evaluates the network once on $(x_t,\,t/T)$, forms the posterior-shaped mean
$$\text{mean}=\frac{1}{\sqrt{\alpha_t}}\Big(x_t-\frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\hat\varepsilon\Big)$$
with the three coefficients read as Python floats from the float64 schedule, and adds $\sqrt{\tilde\beta_t}\,z$ only for $t\ge2$; at $t=1$ it returns the mean itself (so the noise argument is ignored).
`sample_loop` visits $t=T,T-1,\dots,1$ in descending order and uses `noises[t - 1]` at timestep $t$; it rebinds `x` to new tensors and never writes into `x_T` or `noises`.

In [ ]:
def make_schedule(T, beta_1, beta_T):
    if not isinstance(T, int) or isinstance(T, bool) or T < 2:
        raise ValueError("T must be an int >= 2")
    if not (0 < beta_1 <= beta_T < 1):
        raise ValueError("need 0 < beta_1 <= beta_T < 1")
    beta = torch.zeros(T + 1, dtype=torch.float64)
    beta[1:] = torch.linspace(beta_1, beta_T, T, dtype=torch.float64)
    alpha = 1 - beta
    alpha_bar = torch.cumprod(alpha, dim=0)
    beta_tilde = torch.zeros(T + 1, dtype=torch.float64)
    beta_tilde[1:] = (1 - alpha_bar[:-1]) / (1 - alpha_bar[1:]) * beta[1:]
    return {"beta": beta, "alpha": alpha, "alpha_bar": alpha_bar, "beta_tilde": beta_tilde}


def p_sample(model, x_t, t, z, schedule):
    T = schedule["beta"].shape[0] - 1
    if not isinstance(t, int) or isinstance(t, bool) or not 1 <= t <= T:
        raise ValueError("t must be an int in 1..T")
    B = x_t.shape[0]
    eps_hat = model(x_t, torch.full((B, 1), t / T, dtype=x_t.dtype))
    beta_t = schedule["beta"][t].item()
    alpha_t = schedule["alpha"][t].item()
    alpha_bar_t = schedule["alpha_bar"][t].item()
    mean = (x_t - beta_t / math.sqrt(1 - alpha_bar_t) * eps_hat) / math.sqrt(alpha_t)
    if t == 1:
        return mean
    return mean + math.sqrt(schedule["beta_tilde"][t].item()) * z


def sample_loop(model, x_T, noises, schedule):
    T = schedule["beta"].shape[0] - 1
    if tuple(noises.shape) != (T, *x_T.shape):
        raise ValueError("noises must have shape (T, *x_T.shape)")
    x = x_T
    for t in range(T, 0, -1):
        x = p_sample(model, x, t, noises[t - 1], schedule)
    return x


schedule = make_schedule(5, 0.1, 0.5)

### Probes 1–2: single steps

In [ ]:
step5 = p_sample(ToyEps(), XT2, 5, Z2, schedule)
step2 = p_sample(ToyEps(), XT2, 2, Z2, schedule)
step1 = p_sample(ToyEps(), XT2, 1, Z2, schedule)
step1_zero = p_sample(ToyEps(), XT2, 1, torch.zeros_like(Z2), schedule)


def p_sample_noisy_final(model, x_t, t, z, schedule):
    """Defective variant: adds sqrt(beta[1]) * z at t = 1."""
    T = schedule["beta"].shape[0] - 1
    eps_hat = model(x_t, torch.full((x_t.shape[0], 1), t / T, dtype=x_t.dtype))
    mean = (x_t - schedule["beta"][t].item() / math.sqrt(1 - schedule["alpha_bar"][t].item()) * eps_hat) \
        / math.sqrt(schedule["alpha"][t].item())
    if t == 1:
        return mean + math.sqrt(schedule["beta"][1].item()) * z
    return mean + math.sqrt(schedule["beta_tilde"][t].item()) * z


step1_bug = p_sample_noisy_final(ToyEps(), XT2, 1, Z2, schedule)
print("t=5:", step5.tolist())
print("t=2:", step2.tolist())
print("t=1:", step1.tolist(), " noisy-final diff:", (step1_bug - step1).abs().max().item())

### Probes 3–5: the loop

In [ ]:
X_T_clone, NOISES_clone = X_T.clone(), NOISES.clone()
final = sample_loop(ToyEps(), X_T, NOISES, schedule)


class Recorder(nn.Module):
    def __init__(self):
        super().__init__()
        self.visited = []

    def forward(self, x_t, t_scaled):
        self.visited.append(t_scaled[0, 0].item())
        return 0.5 * x_t + t_scaled


recorder = Recorder()
final_recorded = sample_loop(recorder, X_T, NOISES, schedule)


def sample_loop_ascending(model, x_T, noises, schedule):
    T = schedule["beta"].shape[0] - 1
    x = x_T
    for t in range(1, T + 1):
        x = p_sample(model, x, t, noises[t - 1], schedule)
    return x


final_ascending = sample_loop_ascending(ToyEps(), X_T, NOISES, schedule)

noises_first_changed = NOISES.clone()
noises_first_changed[0] = 100 * torch.ones(2, 2)
final_first_changed = sample_loop(ToyEps(), X_T, noises_first_changed, schedule)
noises_last_changed = NOISES.clone()
noises_last_changed[4] = 100 * torch.ones(2, 2)
final_last_changed = sample_loop(ToyEps(), X_T, noises_last_changed, schedule)


def raises_value_error(fn):
    try:
        fn()
    except ValueError:
        return True
    return False


invalid = [
    raises_value_error(lambda: p_sample(ToyEps(), XT2, 0, Z2, schedule)),
    raises_value_error(lambda: p_sample(ToyEps(), XT2, 6, Z2, schedule)),
    raises_value_error(lambda: sample_loop(ToyEps(), X_T, torch.zeros(4, 2, 2), schedule)),
]
print("loop:", final.tolist())
print("visited t/T:", [round(v, 6) for v in recorder.visited])
print("ascending diff:", (final_ascending - final).abs().max().item())
print("invalid:", invalid)

### Why the final step is noiseless

Under this unit's convention $\sigma_1^2=\tilde\beta_1=\frac{1-\bar\alpha_0}{1-\bar\alpha_1}\beta_1=0$ because $\bar\alpha_0=1$, so the reverse step from $x_1$ to $x_0$ has zero variance and its output is exactly the predicted mean; returning `mean` makes that exact rather than relying on multiplying `z` by $0$.
Adding noise at $t=1$ (for example $\sqrt{\beta_1}\,z$) would blur every final sample by noise that no later step removes, which is exactly what the noisy-final variant shows.

### Answer check

In [ ]:
# Probe 1
assert torch.allclose(step5, torch.tensor([[0.2857176960, -2.2329156399], [0.7895655632, 0.5145742893]]), atol=ATOL, rtol=RTOL)
assert torch.allclose(step2, torch.tensor([[0.4609285593, -1.1616884470], [1.0526943207, 0.3654916286]]), atol=ATOL, rtol=RTOL)
# Probe 2
assert torch.allclose(step1, torch.tensor([[0.2883037031, -0.7766073942], [0.9982443452, -0.0666666701]]), atol=ATOL, rtol=RTOL)
assert torch.equal(step1, step1_zero)
assert (step1_bug - step1).abs().max().item() > 0.1
# Probe 3
assert torch.allclose(final, torch.tensor([[-0.4030713737, -2.0024237633], [-1.2903687954, 0.1871905029]]), atol=ATOL, rtol=RTOL)
assert torch.allclose(torch.tensor(recorder.visited), torch.tensor([1.0, 0.8, 0.6, 0.4, 0.2]), atol=1e-7, rtol=0)
assert torch.equal(final_recorded, final)
assert (final_ascending - final).abs().max().item() > 0.1
# Probe 4
assert torch.equal(final_first_changed, final)
assert not torch.equal(final_last_changed, final)
# Probe 5
assert torch.equal(X_T, X_T_clone) and torch.equal(NOISES, NOISES_clone)
assert invalid == [True, True, True]
assert final.shape == X_T.shape and final.dtype == X_T.dtype